# ICML 2013 Black Box Learning Challenge - CNN with ResNet Solution

## Alternative Approach
- **Model**: Convolutional Neural Network with Residual Connections
- **Benefits**: Faster training, fewer parameters, good for image-like data
- **Strategy**: 
  1. Treat data as 2D images (if possible)
  2. Use data augmentation for better generalization
  3. Apply ensemble learning with cross-validation
  4. Use test-time augmentation for final predictions

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name()}")

## 2. Load and Analyze Data

In [ ]:
train_df = pd.read_csv('train.csv')
X_train = train_df.iloc[:, :-1].values.astype(np.float32)
y_train = train_df.iloc[:, -1].values.astype(np.int64)

test_df = pd.read_csv('test.csv')
X_test = test_df.values.astype(np.float32)

print("Training Data:")
print(f"  Shape: {X_train.shape}")
print(f"  Samples: {X_train.shape[0]}")
print(f"  Features: {X_train.shape[1]}")
print(f"  Classes: {len(np.unique(y_train))}")
print(f"  Class distribution: {np.bincount(y_train)}")

print("\nTest Data:")
print(f"  Shape: {X_test.shape}")

In [ ]:
n_features = X_train.shape[1]
sqrt_feat = int(np.sqrt(n_features))

if sqrt_feat * sqrt_feat == n_features:
    print(f"Data can be reshaped to: {sqrt_feat}x{sqrt_feat}")
    IS_IMAGE_DATA = True
    IMG_SIZE = sqrt_feat
else:
    print(f"Data is not square. Finding best rectangular shape...")
    factors = [(i, n_features // i) for i in range(1, int(np.sqrt(n_features)) + 1) if n_features % i == 0]
    h, w = min(factors, key=lambda x: abs(x[0] - x[1]))
    print(f"Best shape: {h}x{w}")
    IS_IMAGE_DATA = True
    IMG_SIZE = (h, w)

if IS_IMAGE_DATA and isinstance(IMG_SIZE, int) and IMG_SIZE <= 32:
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for i, ax in enumerate(axes.flat):
        img = X_train[i].reshape(IMG_SIZE, IMG_SIZE)
        ax.imshow(img, cmap='gray')
        ax.set_title(f'Label: {y_train[i]}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## 3. Data Preprocessing and Augmentation

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data normalized using StandardScaler")
print(f"  Train - Mean: {X_train_scaled.mean():.6f}, Std: {X_train_scaled.std():.6f}")

In [ ]:
class AugmentedDataset(Dataset):
    """Dataset with random augmentations"""
    
    def __init__(self, X, y, img_size, augment=True):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
        self.img_size = img_size
        self.augment = augment
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        
        if isinstance(self.img_size, int):
            x = x.view(1, self.img_size, self.img_size)
        else:
            x = x.view(1, self.img_size[0], self.img_size[1])
        
        if self.augment:
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[2])
            
            if torch.rand(1) > 0.5:
                x = torch.flip(x, dims=[1])
            
            if torch.rand(1) > 0.7:
                noise = torch.randn_like(x) * 0.05
                x = x + noise
        
        return x, y

print("Augmented dataset class defined!")

## 4. ResNet-style CNN Architecture

In [ ]:
class ResidualBlock(nn.Module):
    """Residual block with skip connection"""
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1,
                         stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class ResNetClassifier(nn.Module):
    """ResNet-style CNN for classification"""
    
    def __init__(self, num_classes, img_size, channels=[32, 64, 128], dropout=0.5):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, channels[0], kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels[0])
        
        self.layer1 = self._make_layer(channels[0], channels[0], 2)
        self.layer2 = self._make_layer(channels[0], channels[1], 2, stride=2)
        self.layer3 = self._make_layer(channels[1], channels[2], 2, stride=2)
        
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(channels[2], num_classes)
    
    def _make_layer(self, in_channels, out_channels, num_blocks, stride=1):
        layers = []
        layers.append(ResidualBlock(in_channels, out_channels, stride))
        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(out_channels, out_channels))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.fc(x)
        return x

print("ResNet-style CNN architecture defined!")

## 5. Configuration

In [ ]:
num_classes = len(np.unique(y_train))

CONFIG = {
    'img_size': IMG_SIZE,
    'num_classes': num_classes,
    'channels': [32, 64, 128],
    'dropout': 0.5,
    
    'epochs': 80,
    'batch_size': 128,
    'lr': 0.001,
    'weight_decay': 1e-4,
    
    'n_folds': 5,
    'use_kfold': True,
    'use_tta': True,
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

## 6. Training Function

In [ ]:
def train_model(model, train_loader, val_loader, epochs, lr, weight_decay, device):
    """Train a single model"""
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5, verbose=True
    )
    
    best_val_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        for x, y in tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}', leave=False):
            x, y = x.to(device), y.to(device)
            
            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_correct += predicted.eq(y).sum().item()
            train_total += y.size(0)
        
        train_acc = 100.0 * train_correct / train_total
        avg_train_loss = train_loss / len(train_loader)
        
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                outputs = model(x)
                loss = criterion(outputs, y)
                
                val_loss += loss.item()
                _, predicted = outputs.max(1)
                val_correct += predicted.eq(y).sum().item()
                val_total += y.size(0)
        
        val_acc = 100.0 * val_correct / val_total
        avg_val_loss = val_loss / len(val_loader)
        
        scheduler.step(val_acc)
        
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        if (epoch + 1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1}: Train Acc={train_acc:.2f}%, Val Acc={val_acc:.2f}%, "
                  f"Train Loss={avg_train_loss:.4f}, Val Loss={avg_val_loss:.4f}")
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
    
    return model, best_val_acc, history

print("Training function defined!")

## 7. Cross-Validation Training

In [ ]:
if CONFIG['use_kfold']:
    print("="*70)
    print(f"TRAINING WITH {CONFIG['n_folds']}-FOLD CROSS-VALIDATION")
    print("="*70)
    
    skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=SEED)
    fold_models = []
    fold_accuracies = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train)):
        print(f"\nFold {fold+1}/{CONFIG['n_folds']}")
        print("-" * 50)
        
        X_train_fold = X_train_scaled[train_idx]
        y_train_fold = y_train[train_idx]
        X_val_fold = X_train_scaled[val_idx]
        y_val_fold = y_train[val_idx]
        
        train_dataset = AugmentedDataset(X_train_fold, y_train_fold, CONFIG['img_size'], augment=True)
        val_dataset = AugmentedDataset(X_val_fold, y_val_fold, CONFIG['img_size'], augment=False)
        
        train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], 
                                 shuffle=True, num_workers=2, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'],
                               shuffle=False, num_workers=2, pin_memory=True)
        
        model = ResNetClassifier(
            num_classes=CONFIG['num_classes'],
            img_size=CONFIG['img_size'],
            channels=CONFIG['channels'],
            dropout=CONFIG['dropout']
        ).to(device)
        
        model, best_acc, history = train_model(
            model, train_loader, val_loader,
            CONFIG['epochs'], CONFIG['lr'], CONFIG['weight_decay'], device
        )
        
        fold_models.append(model)
        fold_accuracies.append(best_acc)
        
        print(f"\nFold {fold+1} Best Val Acc: {best_acc:.2f}%")
    
    print("\n" + "="*70)
    print(f"Cross-Validation Results:")
    print(f"  Mean Accuracy: {np.mean(fold_accuracies):.2f}%")
    print(f"  Std Accuracy: {np.std(fold_accuracies):.2f}%")
    print(f"  Individual Folds: {[f'{acc:.2f}%' for acc in fold_accuracies]}")
    print("="*70)
    
else:
    print("Training single model (no cross-validation)...")
    from sklearn.model_selection import train_test_split
    
    X_train_split, X_val, y_train_split, y_val = train_test_split(
        X_train_scaled, y_train, test_size=0.15, stratify=y_train, random_state=SEED
    )
    
    train_dataset = AugmentedDataset(X_train_split, y_train_split, CONFIG['img_size'], augment=True)
    val_dataset = AugmentedDataset(X_val, y_val, CONFIG['img_size'], augment=False)
    
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                             shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'],
                           shuffle=False, num_workers=2, pin_memory=True)
    
    model = ResNetClassifier(
        num_classes=CONFIG['num_classes'],
        img_size=CONFIG['img_size'],
        channels=CONFIG['channels'],
        dropout=CONFIG['dropout']
    ).to(device)
    
    model, best_acc, history = train_model(
        model, train_loader, val_loader,
        CONFIG['epochs'], CONFIG['lr'], CONFIG['weight_decay'], device
    )
    
    fold_models = [model]
    fold_accuracies = [best_acc]
    
    print(f"\nBest Validation Accuracy: {best_acc:.2f}%")

## 8. Generate Predictions with Ensemble and TTA

In [ ]:
def predict_with_tta(models, X, img_size, device, num_tta=5):
    """Predict with test-time augmentation"""
    
    all_predictions = []
    
    for model in models:
        model.eval()
    
    with torch.no_grad():
        for tta_iter in range(num_tta):
            augment = (tta_iter > 0) and CONFIG['use_tta']
            
            test_dataset = AugmentedDataset(
                X, np.zeros(len(X), dtype=np.int64), img_size, augment=augment
            )
            test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
            
            for model in models:
                preds = []
                for x, _ in test_loader:
                    x = x.to(device)
                    outputs = model(x)
                    probs = F.softmax(outputs, dim=1)
                    preds.append(probs.cpu().numpy())
                
                all_predictions.append(np.vstack(preds))
    
    avg_probs = np.mean(all_predictions, axis=0)
    final_predictions = np.argmax(avg_probs, axis=1)
    
    return final_predictions

print("="*70)
print("GENERATING PREDICTIONS")
print("="*70)

num_tta = 5 if CONFIG['use_tta'] else 1
print(f"Using {len(fold_models)} models with {num_tta} TTA iterations")

predictions = predict_with_tta(
    fold_models, X_test_scaled, CONFIG['img_size'], device, num_tta
)

print(f"\nGenerated {len(predictions)} predictions")

## 9. Create Submission File

In [ ]:
submission = pd.DataFrame({
    'Id': range(len(predictions)),
    'Prediction': predictions
})

submission.to_csv('submission_cnn.csv', index=False)

print("Submission file created: submission_cnn.csv")
print("\nFirst 10 predictions:")
print(submission.head(10))

print("\nPrediction distribution:")
for class_id, count in enumerate(np.bincount(predictions)):
    print(f"  Class {class_id}: {count} ({count/len(predictions)*100:.1f}%)")

## 10. Summary

In [ ]:
total_params = sum(p.numel() for p in fold_models[0].parameters())

print("="*70)
print("SUMMARY")
print("="*70)
print(f"\nModel: ResNet-style CNN")
print(f"  Total parameters: {total_params:,}")
print(f"  Architecture: {CONFIG['channels']}")
print(f"  Dropout: {CONFIG['dropout']}")

print(f"\nEnsemble:")
print(f"  Number of models: {len(fold_models)}")
print(f"  Cross-validation: {CONFIG['use_kfold']}")
print(f"  Test-time augmentation: {CONFIG['use_tta']}")

if CONFIG['use_kfold']:
    print(f"\nValidation Accuracy:")
    print(f"  Mean: {np.mean(fold_accuracies):.2f}%")
    print(f"  Std: {np.std(fold_accuracies):.2f}%")

print("\n" + "="*70)
print("Ready to submit to Kaggle!")
print("="*70)